# Diaspora Event SDK Demo

The Diaspora Event SDK provides Python access to the [Diaspora Event Fabric](https://github.com/globus-labs/diaspora-event-sdk) — a distributed event streaming platform backed by Apache Kafka with Globus Auth identity management and AWS MSK.

This notebook covers:
1. **Authentication** — confidential client, native app login, and advanced GlobusApp injection
2. **Client SDK methods** — user, key, namespace, and topic management
3. **Kafka produce & consume** — with an explanation of how MSK IAM auth works under the hood
4. **Cleanup** — removing resources

For the REST-based consumer API (Confluent Kafka REST Proxy v2 compatible), see [ConsumerRESTDemo.ipynb](ConsumerRESTDemo.ipynb).

## Code provenance

| Module | Source | License |
|--------|--------|---------|
| `sdk/auth/` | [globus-compute SDK](https://github.com/globus/globus-compute/tree/68b1174cc06d4e91f49663128951a0c0080abef9/compute_sdk/globus_compute_sdk/sdk/auth) | Apache 2.0 |
| `sdk/aws_iam_msk.py` | [aws-msk-iam-sasl-signer-python](https://github.com/aws/aws-msk-iam-sasl-signer-python) | Apache 2.0 |
| `sdk/botocore/` | [botocore](https://github.com/boto/botocore) (vendored subset) | Apache 2.0 |

## 1. Authentication

The SDK supports two authentication modes, both handled automatically by `globus_sdk.GlobusApp`:

### 1a. Confidential Client (service account)

Set `DIASPORA_SDK_CLIENT_ID` and `DIASPORA_SDK_CLIENT_SECRET` environment variables. The SDK creates a `ClientApp` for non-interactive server-to-server auth.

In [ ]:
import os

from diaspora_event_sdk import get_globus_app

# Set client credentials (replace with your own or source from a secrets file)
os.environ["DIASPORA_SDK_CLIENT_ID"] = "YOUR_CLIENT_ID"
os.environ["DIASPORA_SDK_CLIENT_SECRET"] = (
    "YOUR_CLIENT_SECRET"  # pragma: allowlist secret
)

app = get_globus_app()
print(f"App type: {type(app).__name__}")
print(f"Refresh tokens enabled: {app.config.request_refresh_tokens}")

### 1b. Native App (interactive user login)

When no client credentials are set, the SDK creates a `UserApp` that triggers a browser-based login flow. This is the default for notebooks and CLI usage.

In [ ]:
from diaspora_event_sdk import DEFAULT_CLIENT_ID, get_globus_app  # noqa: E402

# Remove client credentials to trigger native app flow
for key in ["DIASPORA_SDK_CLIENT_ID", "DIASPORA_SDK_CLIENT_SECRET"]:
    os.environ.pop(key, None)

user_app = get_globus_app()
print(f"App type: {type(user_app).__name__}")
print(f"Default native app client ID: {DEFAULT_CLIENT_ID}")
# Note: calling methods that need auth will open a browser login

### 1c. Passing a GlobusApp to Client

You can pre-configure a `GlobusApp` and pass it to `Client` for full control. The `app` and `authorizer` parameters are mutually exclusive (following the [globus-compute SDK pattern](https://github.com/globus/globus-compute/blob/68b1174cc06d4e91f49663128951a0c0080abef9/compute_sdk/globus_compute_sdk/sdk/client.py#L168-L242)).

```python
# Option A: Let Client create the app automatically (default)
client = Client()

# Option B: Pass a pre-configured app
client = Client(app=my_globus_app)

# Option C: Pass a raw authorizer (advanced, no auto-refresh)
client = Client(authorizer=my_authorizer)
```

### 1d. Token Storage

Tokens are stored in `~/.diaspora/storage.db` using `SQLiteTokenStorage` (globus-sdk v4). The namespace isolates tokens by environment and auth mode:
- User login: `user/{environment}`
- Client credentials: `clientprofile/{environment}/{client_id}`

In [ ]:
from diaspora_event_sdk.sdk.auth.token_storage import (  # noqa: E402
    _get_storage_filepath,
    _resolve_namespace,
)

print(f"Storage file: {_get_storage_filepath()}")
print(f"Current namespace: {_resolve_namespace()}")

## 2. Client SDK Methods

In [ ]:
%pip install -e '../../.[kafka-python]'

import json
import uuid
from datetime import datetime

from diaspora_event_sdk import Client
from diaspora_event_sdk.sdk.kafka_client import KafkaConsumer, KafkaProducer

c = Client()
print(f"Subject: {c.subject_openid}")
print(f"Namespace: {c.namespace}")

In [ ]:
# 2a. Create User (optional — create_key will auto-create if needed)
user_result = c.create_user()
print(json.dumps(user_result, indent=2, default=str))

In [ ]:
# 2b. Create Key — returns AWS IAM credentials + Kafka bootstrap endpoint
key_result = c.create_key()
print(json.dumps(key_result, indent=2, default=str))

In [ ]:
# 2c. List Namespaces
namespaces_result = c.list_namespaces()
print(json.dumps(namespaces_result, indent=2, default=str))

In [ ]:
# 2d. Create Topic
topic_name = f"topic-{str(uuid.uuid4())[:5]}"
create_topic_result = c.create_topic(topic_name)
print(json.dumps(create_topic_result, indent=2, default=str))

kafka_topic = f"{c.namespace}.{topic_name}"
print(f"\nKafka topic name: {kafka_topic}")

## 3. Kafka Produce & Consume

### How Kafka authentication works internally

When you call `create_key()`, the service returns **AWS IAM credentials** (access key + secret key) and an **MSK bootstrap endpoint**. The SDK stores these in environment variables (`OCTOPUS_AWS_ACCESS_KEY_ID`, `OCTOPUS_AWS_SECRET_ACCESS_KEY`, `OCTOPUS_BOOTSTRAP_SERVERS`).

When `KafkaProducer` or `KafkaConsumer` connects to the MSK cluster, the `MSKTokenProvider` generates a **SASL/OAUTHBEARER token** by:
1. Creating an AWS SigV4 presigned URL to `https://kafka.{region}.amazonaws.com/?Action=kafka-cluster:Connect`
2. Base64-encoding the signed URL — this becomes the SASL token

The signing uses a vendored subset of [botocore](https://github.com/boto/botocore) (no boto3 dependency). Tokens expire after 15 minutes and are regenerated automatically on each connection.

In [ ]:
# 3a. Produce Messages
p = KafkaProducer(kafka_topic)
for i in range(3):
    message = {
        "message_id": i + 1,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "content": f"Message {i + 1}",
    }
    future = p.send(kafka_topic, message)
    result = future.get(timeout=30)
    print(f"Produced message {i + 1}: offset={result.offset}")
p.close()

In [ ]:
# 3b. Consume Messages
consumer = KafkaConsumer(kafka_topic, auto_offset_reset="earliest")
messages = consumer.poll(timeout_ms=10000)
for tp, msgs in messages.items():
    for message in msgs:
        data = json.loads(message.value.decode("utf-8"))
        print(f"Consumed: {data}")
consumer.close()

## 4. Topic Management

In [ ]:
# 4a. Recreate Topic (delete + recreate, clearing all messages)
recreate_result = c.recreate_topic(topic_name)
print(json.dumps(recreate_result, indent=2, default=str))

In [ ]:
# 4b. Delete Topic
delete_topic_result = c.delete_topic(topic_name)
print(json.dumps(delete_topic_result, indent=2, default=str))

## 5. Cleanup

In [ ]:
# 5a. Delete Key (removes IAM access key; topics + namespace preserved)
delete_key_result = c.delete_key()
print(json.dumps(delete_key_result, indent=2, default=str))

In [ ]:
# 5b. Delete User (full cleanup: user, keys, policies, topics, namespace — irreversible)
delete_user_result = c.delete_user()
print(json.dumps(delete_user_result, indent=2, default=str))